### Imports

In [1]:
import numpy as np
import math
from scipy.stats import chisquare

### Part 1

In [2]:
def get_g(i, A):
    #Target density function g(i) = A^i / i! (proportional to Poisson)
    return (A**i) / math.factorial(i)

def metropolis_hastings_poisson(A, m, n_samples=20000, burn_in=10000):
    #Generates samples from a truncated Poisson distribution using the Metropolis-Hastings algorithm.
    
    samples = []
    current = 0  # Initial state X0
    
    total_iterations = n_samples + burn_in
    
    for i in range(total_iterations):
        # Propose a candidate state Y = X + Delta
        perturbation = np.random.choice([-1, 1])
        proposal = current + perturbation
        
        # Check boundaries [0, m]
        if 0 <= proposal <= m:
            # Calculate acceptance probability alpha = min(1, g(y)/g(x))
            g_y = get_g(proposal, A)
            g_x = get_g(current, A)
            alpha = min(1, g_y / g_x)
            
            # Acceptance decision
            if np.random.rand() < alpha:
                current = proposal # Accept Y: X_{i+1} = Y
            # Else: Reject Y: X_{i+1} = X (remains at current state)
        
        # Discard burn-in period and store samples
        if i >= burn_in:
            samples.append(current)
            
    return np.array(samples)

# Parameters
A = 8.0  #mean_inter_arrival_time * mean_service_time  = 1*8
m = 10 

# Generate samples
samples = metropolis_hastings_poisson(A, m)

# Chi-square goodness-of-fit test
states = np.arange(m + 1)
theoretical_weights = np.array([get_g(i, A) for i in states])
theoretical_dist = theoretical_weights / np.sum(theoretical_weights)

observed_freq = np.array([np.sum(samples == i) for i in states])
expected_freq = theoretical_dist * len(samples)

chi2_stat, p_val = chisquare(f_obs=observed_freq, f_exp=expected_freq)

# Print results
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_val:.4f}")

Chi-square statistic: 70.7761
p-value: 0.0000


### Part 2

In [3]:
A1, A2, m = 4, 4, 10
n_samples, burn_in = 100000, 10000

def g2(i, j):
    if i < 0 or j < 0 or i + j > m:
        return 0
    return (A1**i / math.factorial(i)) * (A2**j / math.factorial(j))

def mh_step(x, y, dx, dy):
    # x+dx, y+dy
    xi, yj = x + dx, y + dy
    r = g2(xi, yj) / g2(x, y) if g2(x, y) > 0 else 0
    if np.random.uniform() < min(1, r):
        return xi, yj
    return x, y

def run(update):
    x, y = 0, 0
    samples = []
    for k in range(n_samples + burn_in):
        x, y = update(x, y)
        if k >= burn_in:
            samples.append((x, y))
    return samples

def update_a(x, y):
    return mh_step(x, y, np.random.choice([-1, 1]), np.random.choice([-1, 1]))

def update_b(x, y):
    x, y = mh_step(x, y, np.random.choice([-1, 1]), 0)
    x, y = mh_step(x, y, 0, np.random.choice([-1, 1]))
    return x, y

def sample_cond(A, max_val):
    # sample from truncated Poisson
    probs = []
    for k in range(max_val + 1):
        probs.append(A**k / math.factorial(k))
    Z = sum(probs)
    norm_probs = []
    for p in probs:
        norm_probs.append(p / Z)
    return np.random.choice(range(max_val + 1), p=norm_probs)

def update_c(x, y):
    #  sample from conditional distributions P(i|j) and P(j|i)
    x = sample_cond(A1, m - y)
    y = sample_cond(A2, m - x)
    return x, y

states = []
for i in range(m + 1):
    for j in range(m + 1):
        if i + j <= m:
            states.append((i, j))

Z = sum(g2(i, j) for i, j in states)
p_teo = []
for (i, j) in states:
    p_teo.append(g2(i, j) / Z)

for title, update in [("MH direct", update_a), ("MH coordinatewise", update_b), ("Gibbs", update_c)]:
    samples = run(update)
    
    f_obs = []
    for state in states:
        f_obs.append(sum(1 for s in samples if s == state))
    
    f_exp = []
    for p in p_teo:
        f_exp.append(p * n_samples)
    
    stat, p_value = chisquare(f_obs=f_obs, f_exp=f_exp)
    print(f"{title}: chi-square = {stat:.4f}, p-value = {p_value:.4f}")

MH direct: chi-square = 90336.3368, p-value = 0.0000
MH coordinatewise: chi-square = 81.8932, p-value = 0.0768
Gibbs: chi-square = 42.6159, p-value = 0.9857


In [4]:
rho = 0.5
n = 10

log_mean = np.random.normal(0, 1)
log_variance = np.random.normal(rho * log_mean, np.sqrt(1 - rho**2))

mean = np.exp(log_mean)
variance = np.exp(log_variance)

X = np.random.normal(loc=mean, scale=np.sqrt(variance), size=n)
print(f"Observations: {X}")

Observations: [ 0.14356589  0.70032285 -0.45241827  0.51040549 -0.81632689  0.70821441
  0.01745445  0.38124323  1.01013484  1.10614668]


In [5]:
def likelihood(mean, var, X):
    if var <= 0:
        return -np.inf
    return np.sum(-0.5 * np.log(2 * np.pi * var) - (X - mean)**2 / (2 * var))

def prior(mean, var):
    if mean <= 0 or var <= 0:
        return -np.inf
    lm, lv = np.log(mean), np.log(var)
    return -(lm**2 - 2*rho*lm*lv + lv**2) / (2*(1-rho**2)) - lm - lv

def posterior(mean, var, X):
    return likelihood(mean, var, X) + prior(mean, var)

In [6]:
def run_mh(X, mean_init, var_init, n_samples=100000, burn_in=10000, step=0.1):
    samples_mean, samples_var = [], []
    m, v = mean_init, var_init
    for k in range(n_samples + burn_in):
        m_new = m + np.random.normal(0, step)
        v_new = v + np.random.normal(0, step)
        r = np.exp(posterior(m_new, v_new, X) - posterior(m, v, X))
        if np.random.uniform() < min(1, r):
            m, v = m_new, v_new
        if k >= burn_in:
            samples_mean.append(m)
            samples_var.append(v)
    return np.mean(samples_mean), np.mean(samples_var)

est_mean, est_var = run_mh(X, mean, variance)

In [7]:
for n in [100, 1000]:
    X_n = np.random.normal(loc=mean, scale=np.sqrt(variance), size=n)
    est_mean, est_var = run_mh(X_n, mean, variance)
    print("N =", n, "Mean:",est_mean,"Variance:",est_var)

print("Mean:", mean, "Variance:", variance)

N = 100 Mean: 0.3397274585900443 Variance: 0.4819629540597071
N = 1000 Mean: 0.29423134739239537 Variance: 0.4374340751700365
Mean: 0.32199285056601684 Variance: 0.43335200813398456
